# Evaluation — Finetuned Qwen3-ASR

In [ ]:
!pip install -U "transformers>=5.13.0" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec

## Setup (Load the Written-Out Checkpoint)

In [ ]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

CHECKPOINT_DIR = "/training-1/asr_finetune/checkpoint"   # <-- adapt
processor = AutoProcessor.from_pretrained(CHECKPOINT_DIR)
model = AutoModelForMultimodalLM.from_pretrained(CHECKPOINT_DIR, device_map="auto")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

def transcribe(audio_path, prompt=None, language=None):
    kwargs = {"audio": audio_path}
    if prompt:
        kwargs["prompt"] = prompt
    if language:
        kwargs["language"] = language
    inputs = processor.apply_transcription_request(**kwargs).to(model.device, model.dtype)

    output_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    return processor.decode(generated_ids, return_format="transcription_only")[0].strip()

## Kaldi Test Data

In [ ]:
import torchaudio
from IPython.display import Audio, display


dataset = []

dataset_path = "/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test" #other languages *_de *_fr *_es
# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/en/test /de/ /fr/ /es/
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/en/test /de/ /fr/ /es/ 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test /de/ /fr/ /es/
# Voxpopuli: /training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test *_de *_fr *_es

wav_scp_dict = {}
with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
    for line in rf:
        id, wav_path = line.strip().split("\t")
        wav_scp_dict[id] = wav_path

text_dict = {}
with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
    for line in rf:
        id, text = line.strip().split("\t")
        text_dict[id] = text

for id, wav_path in wav_scp_dict.items():
    text = text_dict[id]
    dataset.append({"audio" : wav_path, "text" : text})

#helper function: load audio and make sure it is at 16kHz
def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform
    
first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = load_audio(first_entry["audio"])
display(Audio(audio, rate=16000))


## Metrics (WER, CER, Error Counts)

In [ ]:
import jiwer, re, string

def normalize(text):
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()

def compute_metrics(reference, hypothesis):
    ref, hyp = normalize(reference), normalize(hypothesis)
    p = jiwer.process_words(ref, hyp)
    return {"wer": jiwer.wer(ref, hyp), "cer": jiwer.cer(ref, hyp),
            "substitutions": p.substitutions, "deletions": p.deletions,
            "insertions": p.insertions}

def word_errors(reference, hypothesis):
    ref, hyp = normalize(reference), normalize(hypothesis)
    out = []
    for c in jiwer.process_words(ref, hyp).alignments[0].chunks:
        r = " ".join(ref.split()[c.ref_start_idx:c.ref_end_idx])
        h = " ".join(hyp.split()[c.hyp_start_idx:c.hyp_end_idx])
        if c.type != "equal":
            out.append((c.type, r, h))
    return out

## Run Inference on the Test Set

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

rows = []
for sample in tqdm(dataset):
    hyp = transcribe(sample["audio"])
    rows.append({"audio": sample["audio"], "reference": sample["text"], "hypothesis": hyp,
                 **compute_metrics(sample["text"], hyp)})

results = pd.DataFrame(rows)
results.head()

## Corpus-Level Results

In [ ]:
print("Mean WER:", results["wer"].mean())
print("Mean CER:", results["cer"].mean())
print("Micro WER:", jiwer.wer(" ".join(results["reference"]), " ".join(results["hypothesis"])))
results[["wer", "cer"]].describe()

## Worst Samples — Manual Inspection

In [ ]:
results.sort_values("wer", ascending=False).head(10)

In [ ]:
from IPython.display import Audio, display

for _, row in results.sort_values("wer", ascending=False).head(10).iterrows():
    print("=" * 70)
    print("AUDIO  :", row["audio"])
    print("REF    :", row["reference"])
    print("HYP    :", row["hypothesis"])
    print(f"WER {row['wer']:.2%} | CER {row['cer']:.2%}")
    display(Audio(torchaudio.load(row["audio"], normalize=True)[0], rate=16000))

## Word-Level Error Analysis

In [ ]:
from collections import Counter

errors = Counter()
for _, row in results.iterrows():
    for etype, r, h in word_errors(row["reference"], row["hypothesis"]):
        errors[(etype, r, h)] += 1

pd.DataFrame([(t, r, h, c) for (t, r, h), c in errors.most_common(30)],
             columns=["type", "ref", "hyp", "count"])